#**Proyecto: Integración entre sistemas**
#Evaluación del módulo Integración de datos

## Etapa 1: Lección 1 - Fundamentos de la integración de datos

**Objetivo:** Comprender los principios, beneficios, tipos y buenas prácticas de integración de datos.

### Tipos de integración aplicables al caso de logística inteligente:

* **Integración por consolidación:** para reunir datos históricos y operacionales en un data warehouse centralizado.
* **Integración por replicación:** útil para sincronizar sistemas aislados que aún no pueden migrarse.
* **Integración por virtualización:** para acceso en tiempo real desde dashboards o herramientas analíticas sin mover físicamente los datos.

> En el flujo propuesto se emplea una combinación de consolidación (batch) y streaming, apoyados en Apache NiFi y Kafka. La virtualización será un componente conceptual utilizado para lectura ligera desde herramientas de monitoreo o reporting.

### Mapa Conceptual del Flujo de Integración

1. **Fuentes de datos**

   * CSV operativo desde sistema de ventas (batch)
   * JSON de sensores IoT (streaming)
   * Logs de operaciones en TXT (batch)
   * API REST externa para seguimiento de pedidos (simulación)

2. **Herramientas de ingesta**

   * Apache NiFi: ingesta batch y automatización
   * Apache Kafka: streaming de sensores en tiempo real

3. **Transformaciones**

   * Limpieza de datos
   * Enriquecimiento con reglas de negocio
   * Validación de esquemas

4. **Almacenamiento intermedio**

   * HDFS / Data Lake (simulado como carpeta local o S3)
   * Tópicos Kafka para streaming

5. **Consumo**

   * Dashboard de monitoreo en tiempo real (simulado)
   * Herramienta de BI (futuro)

6. **Automatización y monitoreo**

   * Apache NiFi: gestión de errores, notificaciones, logs de auditoría

### Justificación de la arquitectura

**Problema principal:** Los sistemas actuales están aislados, lo que impide una visión unificada de las operaciones y retrasa la toma de decisiones.

**Decisión técnica:**

* Se utiliza **Apache NiFi** como columna vertebral del flujo, por su capacidad para:

  * Conectar múltiples fuentes
  * Automatizar la integración
  * Aplicar transformaciones ligeras y control de errores

* Se emplea **Apache Kafka** para:

  * Integrar flujos en tiempo real desde sensores IoT
  * Procesar eventos por ventana de tiempo (por minuto/hora)
  * Habilitar consumidores futuros (reportes, alertas, modelos ML)

Esta decisión permite una arquitectura modular, escalable y resiliente para la empresa logística simulada.

---

## Etapa 2: Lección 2 – Ingesta de Datos Batch

**Objetivo:** Implementar flujos de ingesta por lotes utilizando Apache NiFi, simulando datos desde archivos locales, con transformaciones básicas y validaciones.

### Supuestos para la simulación

Se emplean archivos locales en formato CSV, JSON y TXT simulando:

* `ventas.csv` (datos de pedidos)
* `rutas.json` (rutas logísticas)
* `logs.txt` (registros operativos)

### Flujo de ingesta batch en Apache NiFi

**1. Origen de datos:**

* Ingesta de archivos desde carpeta local `/data_batch/input/`

**2. Estructura del flujo:**

```
ListFile → FetchFile → UpdateAttribute
         → RouteOnAttribute (por extensión)
            ├── CSV: ConvertRecord → QueryRecord → ValidateRecord → PutFile (/output/csv)
            └── JSON: JoltTransformJSON → ValidateRecord → PutFile (/output/json)
```

**3. Validaciones y automatización:**

* `ValidateRecord`: verifica integridad de campos
* `PutFile`: escritura de archivos limpios
* Cron cada 60 segundos
* `PutEmail`: alertas en caso de error

**4. Transformaciones aplicadas:**

| Tipo de dato | Acción                   | Componente NiFi   |
| ------------ | ------------------------ | ----------------- |
| CSV          | Filtro por cantidad > 10 | QueryRecord       |
| JSON         | Renombrar campo "origen" | JoltTransformJSON |
| Ambos        | Validación de campos     | ValidateRecord    |

### Resultados esperados:

* Archivos limpios en `/data_batch/output/`
* Archivos con errores son registrados y generan alertas
* Flujo modular y automático listo para ampliación

---

## Etapa 3: Lección 3 - Ingesta de Datos en Streaming

**Objetivo:** Desarrollar un flujo de ingesta en tiempo real usando socket TCP (simulado), con procesamiento por ventanas, enriquecimiento de datos y lógica condicional.

### Flujo implementado en NiFi (simulado)

**1. Origen de datos:**

* Socket TCP en puerto 9999 (Netcat)

**2. Estructura del flujo:**

```
ListenTCP → SplitText → ParseJSON → ValidateRecord → UpdateRecord
          → RouteOnAttribute (por prioridad/tipo_evento)
          → QueryRecord (ventana de 1 minuto)
          → PutFile (/output/stream)
```

**3. Enriquecimiento:**

* Se agrega campo `fecha_evento_actual`
* Clasificación por tipo\_evento: "CRÍTICO", "ALERTA", "NORMAL"

**4. Lógica condicional aplicada:**

| Condición                     | Acción                      |
| ----------------------------- | --------------------------- |
| prioridad == "alta"           | Enviar alerta a consola/log |
| tipo\_evento == "sensor\_iot" | Guardar en ruta específica  |
| mensaje contiene "error"      | Clasificar como CRÍTICO     |

**5. Simulación del flujo**

* Uso de Netcat:

```bash
nc -lk 9999
```

* Ejemplo de mensaje:

```json
{"id_evento": 1001, "tipo_evento": "sensor_iot", "mensaje": "Alta temperatura", "prioridad": "alta"}
```

### Comparativa Batch vs Streaming

| Característica   | Ingesta Batch    | Ingesta Streaming           |
| ---------------- | ---------------- | --------------------------- |
| Origen           | Archivos locales | Socket TCP (en vivo)        |
| Frecuencia       | Por lotes        | Continua                    |
| Transformaciones | Filtros básicos  | Lógica condicional + tiempo |
| Monitoreo        | Logs + email     | Output inmediato            |

---

## Etapa 4: Automatización y Orquestación

**Objetivo:** Unificar y automatizar la arquitectura completa, integrando ingestas batch y streaming con monitoreo, reintentos y control de errores.

### Arquitectura general en Apache NiFi (simulada)

**1. Módulo Batch:**

```
GetFile → DetectDuplicate → ReplaceText → UpdateAttribute
       → RouteOnAttribute → PutFile (/processed/batch)
```

**2. Módulo Streaming:**

```
ListenTCP → ExecuteScript (ventanas)
         → RouteOnAttribute (por anomalía)
         → PutFile (/processed/stream)
```

**3. Automatización y monitoreo:**

```
TimerDriven → RetryFlowFile
→ LogMessage + SendEmail (en errores)
```

### Ventajas

* Plataforma visual, sin código
* Modular, reutilizable y escalable
* Compatible con múltiples fuentes

### Limitaciones

* Requiere hardware dedicado en producción
* Interfaz puede ser compleja a gran escala

---

## Diagrama de Arquitectura Modular - Integración de Datos

```md
FUENTES DE DATOS HETEROGÉNEAS
│
├── Archivo CSV (Ventas)
├── JSON API (Incidentes IoT)
└── Logs del sistema (TXT)
      │
      ▼
INGESTA BATCH (Apache NiFi)
│
├── Scheduler programado
├── Validación de esquema
├── Transformaciones básicas
└── Almacenamiento temporal (Landing Zone)
      │
      ▼
INGESTA STREAMING (Apache Kafka / Simulación)
│
├── Escucha socket / File Listener
├── Enriquecimiento condicional
├── Procesamiento por ventanas
└── Garantías de entrega (at-least-once)
      │
      ▼
PROCESAMIENTO Y ENRUTAMIENTO (NiFi / Kafka Streams)
│
├── Join de flujos batch y streaming
├── Reglas de negocio
├── Transformaciones avanzadas
└── Envío a Data Lake
      │
      ▼
MONITOREO Y AUTOMATIZACIÓN
│
├── Alertas por condición
├── Métricas de procesamiento
├── Webhooks / notificaciones
└── Reintentos y trazabilidad
      │
      ▼
CAPA DE CONSUMO / REPORTING
│
├── Dashboard (Power BI / Grafana)
├── Consolidación para reportes
└── Análisis o entrenamiento ML
```

---

## Conclusiones y Aprendizajes

Durante el desarrollo del proyecto se construyó una arquitectura moderna y modular que aborda múltiples tipos de integración de datos:

* La separación de flujos batch y streaming mejora la claridad y escalabilidad del diseño.
* Apache NiFi resultó ser una herramienta robusta para ingesta, transformación y automatización.
* La incorporación de monitoreo y control de errores desde el inicio incrementó la confiabilidad del sistema.
* El streaming, incluso simulado, evidenció ventajas competitivas para la detección temprana de eventos.

Se reforzaron habilidades en arquitectura de datos, diseño ETL, automatización, monitoreo, procesamiento en tiempo real y documentación técnica.

---

## Flujo implementado (Pseudocódigo NiFi)

**Ingesta Batch:**

```
ListFile → FetchFile → UpdateAttribute → RouteOnAttribute
├── CSV: ConvertRecord → QueryRecord → ValidateRecord → PutFile (/output/csv)
└── JSON: JoltTransformJSON → ValidateRecord → PutFile (/output/json)
```

**Ingesta Streaming:**

```
ListenTCP → SplitText → ParseJSON → UpdateRecord
→ RouteOnAttribute → QueryRecord (ventana 1 min) → PutFile (/output/stream)
```

**Monitoreo y Automatización:**

```
TimerDriven → RetryFlowFile → LogAttribute
→ SendEmail (si error) → LogMessage
```

---

## Guía de Ejecución del Flujo (modo local/simulado)

### Requisitos:

* Apache NiFi 1.21+
* Java 8 o superior
* Netcat (opcional para streaming)
* Archivos de entrada: `ventas.csv`, `rutas.json`, `logs.txt`

### 1. Iniciar Apache NiFi

Acceder vía navegador: `http://localhost:8080/nifi`

### 2. Construir el flujo

Seguir el pseudodiagrama o importar plantilla `.xml` si disponible

### 3. Simular Batch

Colocar los archivos en `/input/batch`:

```csv
id,producto,cantidad,fecha
101,Producto A,12,2024-10-10
102,Producto B,7,2024-10-11
```

### 4. Simular Streaming

Con Netcat:

```bash
nc -lk 9999
```

Y enviar mensajes como:

```json
{"id_evento": 1001, "tipo_evento": "sensor_iot", "mensaje": "Alta temperatura", "prioridad": "alta"}
```

### 5. Ver resultados

* Batch en `/output/csv` y `/output/json`
* Streaming en `/output/stream`
* Logs de NiFi con trazas de errores o alertas
